# Component 1 Training Notebook

This notebook is focused on the training path for Component 1.
It uses the implementation already defined in:

- `src/components/component1_encoder.py`
- `src/components/component1_dann.py`
- `src/training/train_component1_dann.py`

Use this notebook to inspect the 4-domain manifest, build a training loader, run one epoch interactively, or launch the full training script.

## Training Pipeline Summary

This notebook trains **Component 1: the shared image encoder** from the pipeline.

In simple terms, this stage does the following:

1. **Read chest X-ray images from 4 domains**: Montgomery, Shenzhen, TBX11K, and NIH ChestX-ray14.
2. **Run Component 0 preprocessing** so every image is normalized into the same tensor format expected by the pipeline.
3. **Feed the 3-channel 1024x1024 image into the shared encoder**.
4. **Produce an image embedding** with shape `[B, 256, 64, 64]`.
5. **Pool that embedding** into a compact feature vector for domain classification.
6. **Train a DANN head with gradient reversal** so the encoder learns features that are less dependent on dataset/domain style.
7. **Keep the main encoder frozen except for LoRA adapters**, which makes training much cheaper than full fine-tuning.

### Why this matters

The project uses multiple chest X-ray datasets collected in different hospitals and conditions.
If the encoder learns domain-specific shortcuts, later segmentation stages may not generalize well.
The DANN objective pushes the shared embedding to become more **domain-invariant**, which is the main goal of Component 1.

### What the notebook shows

- dataset loading from the external HDD
- per-domain sample counts
- construction of the training loader
- model and optimizer setup
- one interactive training epoch
- optional launch of the full training script

### Practical note

The current config uses the `mock` backend by default so the notebook can run even without the full SAM ViT-H dependency stack.
When the real SAM checkpoint and package are available, switch the encoder backend in `configs/component1_dann.yaml` to `segment_anything`.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

In [ ]:
from collections import Counter
from pprint import pprint
import copy
import json
import subprocess

import torch
from torch.utils.data import DataLoader

from src.core.device import pick_device, describe_device
from src.core.seed import seed_everything
from src.training.train_component1_dann import (
    Component1DomainDataset,
    build_component1_manifest,
    build_model,
    build_optimizer,
    build_weighted_sampler,
    collate_component1_batch,
    load_yaml_config,
    maybe_limit_manifest,
    train_one_epoch,
)


## Load Config Files

In [ ]:
PATHS_CONFIG = REPO_ROOT / "configs" / "paths.yaml"
COMPONENT1_CONFIG = REPO_ROOT / "configs" / "component1_dann.yaml"

paths_cfg = load_yaml_config(PATHS_CONFIG)
component1_cfg = load_yaml_config(COMPONENT1_CONFIG)["component1_dann"]

print("Training config loaded from:", COMPONENT1_CONFIG)
pprint(component1_cfg["training"])


## Build And Inspect The 4-Domain Training Manifest

In [ ]:
manifest = build_component1_manifest(paths_config=PATHS_CONFIG, component1_config=COMPONENT1_CONFIG)
counts = Counter(sample.dataset_id for sample in manifest)

print(json.dumps({"counts": counts, "total": len(manifest)}, indent=2, default=int))


## Optional Notebook Overrides

Use this section to keep the real YAML untouched while you run smaller notebook experiments.

In [ ]:
cfg = copy.deepcopy(component1_cfg)

cfg["training"]["batch_size"] = 2
cfg["training"]["epochs"] = 1
cfg["training"]["num_workers"] = 0
cfg["training"]["limit_per_domain"] = 16
cfg["training"]["device"] = None

cfg["encoder"]["backend"] = cfg["encoder"].get("backend", "mock")
cfg["training"]


## Build Dataset, Sampler, And Loader

In [ ]:
seed_everything(int(cfg["training"]["seed"]))
device = pick_device(cfg["training"].get("device"))
print("device:", describe_device(device))

manifest_small = maybe_limit_manifest(manifest, cfg["training"].get("limit_per_domain"))
dataset = Component1DomainDataset(manifest_small)
sampler = build_weighted_sampler(dataset.samples, cfg["data"]["domain_sampling_weights"])
loader = DataLoader(
    dataset,
    batch_size=int(cfg["training"]["batch_size"]),
    sampler=sampler,
    num_workers=int(cfg["training"]["num_workers"]),
    collate_fn=collate_component1_batch,
)

print("limited samples:", len(dataset))
print("batches per epoch:", len(loader))


## Inspect One Training Batch

In [ ]:
batch = next(iter(loader))
print("x_3ch:", tuple(batch["x_3ch"].shape), batch["x_3ch"].dtype)
print("domain_id:", tuple(batch["domain_id"].shape), batch["domain_id"].dtype)
print("datasets:", batch["dataset_id"])


## Build Model And Optimizer

In [ ]:
model = build_model(cfg).to(device)
optimizer = build_optimizer(model, cfg)

trainable = [name for name, param in model.named_parameters() if param.requires_grad]
print("trainable parameter tensors:", len(trainable))
trainable[:20]


## Run One Interactive Epoch

In [ ]:
metrics = train_one_epoch(
    model,
    loader,
    optimizer,
    epoch=0,
    device=device,
    ramp_epochs=int(cfg["training"]["grl_ramp_epochs"]),
    max_lambda=float(cfg["training"]["max_lambda"]),
)
metrics


## Launch The Full Training Script

Set `DRY_RUN = False` to launch the full script. Leave it as `True` if you only want to verify the dataset counts.

In [ ]:
DRY_RUN = True

cmd = [sys.executable, "-m", "src.training.train_component1_dann"]
if DRY_RUN:
    cmd.append("--dry-run")

result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"training command failed with exit code {result.returncode}")
